# AI-QMS — Phase 2: Data Collection & Preprocessing

Dataset sources locked in for Phase 1: synthetic data (still available via `pipeline/generator.py` for experiments), plus the real collected datasets below, mapped onto the canonical queue-event schema by `pipeline/importers.py`.

| Source | Rows | Canonical mapping (key decisions) |
|---|---|---|
| `data/collected/Hospital Wait  TIme Data.csv` | 5000 | `arrival_ts`=ActualArrivalTime, `called_ts`=TriageCompleteTime, `service_start_ts`=ProviderStartTime, `service_end_ts`=ProviderEndTime, `service_type`=Department, `counter_id`=ProviderID (factorized), institution 1. No queue-length column exists — `queue_length_at_arrival` flagged missing; `facility_occupancy_rate`, `triage_category`, `age_group` kept as passthrough features |
| `data/collected/queue_data.csv` | 560 | `arrival_ts`=arrival_time (`D-M-YYYY H.MM` parse), `called_ts`/`service_start_ts`=start_time, `service_end_ts`=finish_time, `queue_length_at_arrival`=queue_length, institution 2, `service_type`="general". Note: the source's `wait_time` equals `finish−start` (verified 560/560) — the canonical `wait_time_min` is derived as `service_start−arrival` instead; source value kept as `reported_wait_time` passthrough |
| `data/collected/Smart_Queue_Management_Survey.xlsx` | 220 | Self-reported buckets (`How long did you wait…`, `How long did your service take…`, people ahead) → bucket midpoints; open-ended buckets ("More than 30") → upper bound + 10 min. `service_start_ts` = arrival + mid(wait), `called_ts`=service_start (self-report), `service_end_ts` = start + mid(service). `institution_id` factorized from organization type, `service_type`=purpose of visit, `organization_type` passthrough |

Rows failing canonical timestamps (e.g. survey rows missing date/time) are dropped by the pipeline's missing-value policy, same as incomplete services.

In [ ]:
from pipeline.importers import (
    combine,
    import_hospital,
    import_queue_log,
    import_survey,
)
from pipeline.cleaning import run_pipeline
from pipeline.schema import PipelineConfig

sources = [
    (import_hospital, "data/collected/Hospital Wait  TIme Data.csv"),
    (import_queue_log, "data/collected/queue_data.csv"),
    (import_survey, "data/collected/Smart_Queue_Management_Survey.xlsx"),
]

raw = combine(sources)
raw.to_csv("data/raw/queue_events_real.csv", index=False)

clean = run_pipeline(raw, PipelineConfig())
clean.to_csv("data/processed/queue_events_clean.csv", index=False)

print("raw:", len(raw), "-> cleaned:", len(clean))
print(clean["source"].value_counts().to_string())

In [ ]:
required = [
    "institution_id",
    "counter_id",
    "service_type",
    "arrival_ts",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "queue_length_at_arrival",
    "wait_time_min",
    "service_time_min",
]
print("nulls in required columns:", int(clean[required].isna().sum().sum()))
print()
print(clean[["wait_time_min", "service_time_min", "hour_of_day", "queue_length_at_arrival"]].describe().round(2).to_string())
print()
print("service_type values:", sorted(clean["service_type"].unique()))

## Design notes

- **Missing values:** rows missing any of arrival/called/service_start/service_end are dropped — they cannot yield a wait_time target (still-in-service). `queue_length_at_arrival` is flagged (`queue_length_missing`) and zero-filled rather than dropped.
- **Outliers:** wait_time_min ≤ 240, service_time_min ≤ 120, queue_length ≤ 200; winsorized (clipped) by default — switch `winsorize_bounds=False` to drop instead.
- **Time normalization:** all timestamps converted to UTC; features hour_of_day, day_of_week, is_weekend derived from arrival_ts.
- **Categorical encoding:** service_type one-hot (default) or label via `PipelineConfig.encode_categorical`.
- **Source-specific passthroughs** (carried through for later model features, nullable by design): facility_occupancy_rate / triage_category / age_group (hospital), reported_wait_time (queue log), organization_type (survey).
- **Reconciliation:** the synthetic generator remains available for experiments; real collected data is the default source going forward.